In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install entsoe-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.5 MB/s eta 0:00:00


In [3]:
pip install pyyaml pandas numpy requests "autogluon.tabular" scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.4/382.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.7/222.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 6.6 MB/s eta 0:00:00


In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import requests
import time
import re
from entsoe import EntsoePandasClient

# ============= Set Google Drive paths ============
base_path = "/content/drive/My Drive/Colab Notebooks/entsoe_pipeline_optimized123"
os.makedirs(base_path, exist_ok=True)

# Folder for all CSV outputs on Drive
csv_folder = os.path.join(base_path, "entsoe_DB_data_DE")
os.makedirs(csv_folder, exist_ok=True)

# ============== Helper functions ==============
def fetch_with_backoff(url, params, headers=None, max_retries=10, base_delay=2):
    delay = base_delay
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers)
            if resp.status_code == 429:
                print(f"[WARNING] 429 Too Many Requests. Sleeping {delay}s (Attempt {attempt}/{max_retries})")
                time.sleep(delay)
                delay *= 2
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            print(f"[WARNING] Request exception: {e}, attempt {attempt}/{max_retries}, sleeping {delay}s.")
            time.sleep(delay)
            delay *= 2
    raise Exception(f"[ERROR] Exceeded {max_retries} retries for {url} with params={params}.")

def query_scheduled_exchanges_with_retries(client, from_zone, to_zone, start, end, dayahead=True, max_retries=10, base_delay=2):
    delay = base_delay
    for attempt in range(1, max_retries + 1):
        try:
            df = client.query_scheduled_exchanges(
                country_code_from=from_zone,
                country_code_to=to_zone,
                start=start,
                end=end,
                dayahead=dayahead
            )
            return df
        except Exception as e:
            print(f"[WARN] query_scheduled_exchanges {from_zone}->{to_zone} attempt {attempt}/{max_retries}: {e}")
            time.sleep(delay)
            delay *= 2
    raise Exception(f"Failed scheduled_exchanges {from_zone}->{to_zone}")

def run_query_with_retry(query_callable, country, query_name, max_retries=10, base_delay=2):
    delay = base_delay
    for attempt in range(1, max_retries + 1):
        try:
            return query_callable(country)
        except Exception as e:
            print(f"[WARN] {query_name} for {country} attempt {attempt}/{max_retries}: {e}")
            time.sleep(delay)
            delay *= 2
    raise Exception(f"Failed {query_name} for {country}")

def sanitize_table_name(raw_name: str) -> str:
    safe = re.sub(r'[^A-Za-z0-9_]+', '_', raw_name)
    if safe and safe[0].isdigit():
        safe = 't_' + safe
    return safe

def process_api_data(df, default_item_id, table_name):
    if isinstance(df, pd.Series):
        df = df.to_frame()
    if "timestamp" not in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df.index):
            df = df.reset_index().rename(columns={'index': 'timestamp'})
        else:
            print(f"[WARN] {table_name}: no 'timestamp' found.")
            return None
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    if df['timestamp'].dt.tz is None:
        df['timestamp'] = df['timestamp'].dt.tz_localize('UTC').dt.tz_convert("Europe/Brussels")
    else:
        df['timestamp'] = df['timestamp'].dt.tz_convert("Europe/Brussels")
    df['timestamp'] = df['timestamp'].dt.tz_localize(None)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        return df
    df = df.set_index('timestamp')
    df = df.resample("h").mean()
    df = df.interpolate(method='linear').ffill().bfill()
    df = df.reset_index()
    df["item_id"] = default_item_id
    return df

# ============== MAIN ==============
def main():
    with open("/content/drive/My Drive/Colab Notebooks/config_DE.yaml", "r") as f:
        config = yaml.safe_load(f)
    countries = config["countries"]
    borders = config["country_borders"]
    states = config.get("states", [])
    neighbors = ["CH", "BE", "CZ", "FR", "NL", "PL", "DK_1", "DK_2", "AT", "10Y1001A1001A47J", "10YNO-2--------T"]

    start = pd.Timestamp("20250401", tz="Europe/Brussels")
    end = pd.Timestamp("20250427", tz="Europe/Brussels")
    API_KEY = "YOUR_ENTSOE_API_KEY"
    client = EntsoePandasClient(api_key=API_KEY)

    # 1. Cross-border expansions
    expanded_borders = []
    for b in borders:
        try:
            a, c = b.split("-", 1)
            expanded_borders.append((a, c))
            expanded_borders.append((c, a))
        except Exception as e:
            print(f"[WARN] skipping border '{b}': {e}")
    expanded_borders = list(dict.fromkeys(expanded_borders))

    # 2. Cross-border imports/exports
    all_cross_tables = []
    all_cross_dfs = {}
    for country_from, country_to in expanded_borders:
        # Imports
        try:
            print(f"Fetching imports: {country_to} to {country_from}...")
            df_i = query_scheduled_exchanges_with_retries(
                client=client,
                from_zone=country_to,
                to_zone=country_from,
                start=start,
                end=end,
                dayahead=True
            )
            tbl_i_raw = f"{country_from}_imports_{country_to}"
            tbl_i = sanitize_table_name(tbl_i_raw)
            df_i = process_api_data(df_i, default_item_id=country_from, table_name=tbl_i_raw)
            if df_i is not None:
                all_cross_tables.append(tbl_i)
                all_cross_dfs[tbl_i] = df_i
        except Exception as e:
            print(f"[WARN] fail cross import {country_to}->{country_from}: {e}")

        # Exports
        try:
            print(f"Fetching exports: {country_from} to {country_to}...")
            df_e = query_scheduled_exchanges_with_retries(
                client=client,
                from_zone=country_from,
                to_zone=country_to,
                start=start,
                end=end,
                dayahead=True
            )
            tbl_e_raw = f"{country_from}_exports_{country_to}"
            tbl_e = sanitize_table_name(tbl_e_raw)
            df_e = process_api_data(df_e, default_item_id=country_from, table_name=tbl_e_raw)
            if df_e is not None:
                all_cross_tables.append(tbl_e)
                all_cross_dfs[tbl_e] = df_e
        except Exception as e:
            print(f"[WARN] fail cross export {country_from}->{country_to}: {e}")

    # 3. Full queries for config countries (all 4 APIs)
    full_queries = {
        "query_load_forecast": lambda c: client.query_load_forecast(c, start=start, end=end),
        "query_generation_forecast": lambda c: client.query_generation_forecast(c, start=start, end=end),
        "query_wind_and_solar_forecast": lambda c: client.query_wind_and_solar_forecast(c, start=start, end=end, psr_type=None),
        "query_day_ahead_prices": lambda c: client.query_day_ahead_prices(c, start=start, end=end),
    }
    country_query_dfs = {}
    for cc in countries:
        for qn, qfunc in full_queries.items():
            tbl_q_raw = f"{cc}_query_{qn}"
            tbl_q = sanitize_table_name(tbl_q_raw)
            try:
                print(f"[INFO] Running {qn} for {cc}...")
                q_data = run_query_with_retry(qfunc, cc, qn, max_retries=5, base_delay=2)
                q_data = process_api_data(q_data, default_item_id=cc, table_name=tbl_q_raw)
                if q_data is not None:
                    country_query_dfs[tbl_q] = q_data
            except Exception as e:
                print(f"[WARN] failed {qn} for {cc}: {e}")

    # 4. Only these 4 queries for neighbor countries
    for nn in neighbors:
        for qn, qfunc in full_queries.items():
            tbl_q_raw = f"{nn}_query_{qn}"
            tbl_q = sanitize_table_name(tbl_q_raw)
            try:
                print(f"[INFO] Running {qn} for neighbor={nn}...")
                q_data = run_query_with_retry(qfunc, nn, qn, max_retries=5, base_delay=2)
                q_data = process_api_data(q_data, default_item_id=nn, table_name=tbl_q_raw)
                if q_data is not None:
                    country_query_dfs[tbl_q] = q_data
            except Exception as e:
                print(f"[WARN] failed {qn} for neighbor={nn}: {e}")

    # 5. co2map for DE
    de_prod, de_cons = [], []
    prod_url = "https://api.co2map.de/ProductionIntensityHistorical/"
    cons_url = "https://api.co2map.de/ConsumptionIntensityHistorical/"
    segments = pd.date_range(start=start, end=end, freq='QE')  # <- FIXED FutureWarning here!
    segments = list(zip([start] + list(segments), list(segments) + [end]))
    for (seg_s, seg_e) in segments:
        st_s = seg_s.strftime("%Y-%m-%d")
        st_e = seg_e.strftime("%Y-%m-%d")
        params_de = {"state": "DE", "country": "DE", "start": st_s, "end": st_e}
        try:
            print(f"Fetching DE production {st_s}->{st_e}")
            p_json = fetch_with_backoff(prod_url, params=params_de)
            p_list = p_json.get("Production-based Intensity (historical)", [])
            df_p = pd.DataFrame(p_list, columns=["timestamp", "Production_Intensity"])
            df_p["timestamp"] = pd.to_datetime(df_p["timestamp"], utc=True)\
                                    .dt.tz_convert("Europe/Brussels")\
                                    .dt.tz_localize(None)
            de_prod.append(df_p)
        except Exception as e:
            print(f"[WARN] DE production fail: {e}")
        try:
            print(f"Fetching DE consumption {st_s}->{st_e}")
            c_json = fetch_with_backoff(cons_url, params=params_de)
            c_list = c_json.get("Consumption-based Intensity (historical)", [])
            df_c = pd.DataFrame(c_list, columns=["timestamp", "Consumption_Intensity"])
            df_c["timestamp"] = pd.to_datetime(df_c["timestamp"], utc=True)\
                                    .dt.tz_convert("Europe/Brussels")\
                                    .dt.tz_localize(None)
            de_cons.append(df_c)
        except Exception as e:
            print(f"[WARN] DE consumption fail: {e}")
    if de_prod:
        de_prod_df = pd.concat(de_prod).drop_duplicates().sort_values("timestamp").reset_index(drop=True)
    else:
        de_prod_df = pd.DataFrame(columns=["timestamp", "Production_Intensity"])
    if de_cons:
        de_cons_df = pd.concat(de_cons).drop_duplicates().sort_values("timestamp").reset_index(drop=True)
    else:
        de_cons_df = pd.DataFrame(columns=["timestamp", "Consumption_Intensity"])

    # 6. co2map for each state
    state_prod, state_cons = {}, {}
    if states:
        for st_code in states:
            s_pdfs, s_cdfs = [], []
            for (sg_s, sg_e) in segments:
                st_s = sg_s.strftime("%Y-%m-%d")
                st_e = sg_e.strftime("%Y-%m-%d")
                params_st = {"state": st_code, "country": "DE", "start": st_s, "end": st_e}
                try:
                    print(f"Fetching Production {st_code} {st_s}->{st_e}")
                    p_json = fetch_with_backoff(prod_url, params=params_st)
                    p_list = p_json.get("Production-based Intensity (historical)", [])
                    df_p = pd.DataFrame(p_list, columns=["timestamp", "Production_Intensity"])
                    df_p["timestamp"] = pd.to_datetime(df_p["timestamp"], utc=True)\
                                            .dt.tz_convert("Europe/Brussels")\
                                            .dt.tz_localize(None)
                    df_p["item_id"] = st_code
                    s_pdfs.append(df_p)
                except Exception as e:
                    print(f"[WARN] production {st_code} fail: {e}")
                try:
                    print(f"Fetching Consumption {st_code} {st_s}->{st_e}")
                    c_json = fetch_with_backoff(cons_url, params=params_st)
                    c_list = c_json.get("Consumption-based Intensity (historical)", [])
                    df_c = pd.DataFrame(c_list, columns=["timestamp", "Consumption_Intensity"])
                    df_c["timestamp"] = pd.to_datetime(df_c["timestamp"], utc=True)\
                                            .dt.tz_convert("Europe/Brussels")\
                                            .dt.tz_localize(None)
                    df_c["item_id"] = st_code
                    s_cdfs.append(df_c)
                except Exception as e:
                    print(f"[WARN] consumption {st_code} fail: {e}")

            if s_pdfs:
                state_prod[st_code] = pd.concat(s_pdfs).drop_duplicates().sort_values("timestamp").reset_index(drop=True)
            if s_cdfs:
                state_cons[st_code] = pd.concat(s_cdfs).drop_duplicates().sort_values("timestamp").reset_index(drop=True)

    # ========== Merge everything (just like in DB, but with DataFrames) ==========
    # Start with all timestamps
    dfs_for_timestamps = list(all_cross_dfs.values()) + list(country_query_dfs.values()) + [de_prod_df, de_cons_df] + list(state_prod.values()) + list(state_cons.values())
    timestamps = pd.concat([df[["timestamp"]] for df in dfs_for_timestamps if not df.empty]).drop_duplicates().sort_values("timestamp")
    df_de = timestamps.copy()

    # Add cross-border columns
    for tbl, df in all_cross_dfs.items():
        for col in df.columns:
            if col not in ["timestamp", "item_id"]:
                new_col_name = tbl + "_" + str(col)
                new_col_name = re.sub(r'[^A-Za-z0-9_]+', '_', new_col_name)
                df_de = df_de.merge(df[["timestamp", col]], on="timestamp", how="left")
                df_de.rename(columns={col: new_col_name}, inplace=True)

    # Add query columns
    for tbl, df in country_query_dfs.items():
        for col in df.columns:
            if col not in ["timestamp", "item_id"]:
                new_col = tbl + "_" + str(col)
                new_col = re.sub(r'[^A-Za-z0-9_]+', '_', new_col)
                df_de = df_de.merge(df[["timestamp", col]], on="timestamp", how="left")
                df_de.rename(columns={col: new_col}, inplace=True)

    # Add DE intensities
    if not de_prod_df.empty:
        df_de = df_de.merge(de_prod_df.rename(columns={"Production_Intensity": "Generation_Intensity"}), on="timestamp", how="left")
    if not de_cons_df.empty:
        df_de = df_de.merge(de_cons_df.rename(columns={"Consumption_Intensity": "Consumption_Intensity"}), on="timestamp", how="left")

    # Add states intensities
    for st_code in states:
        if st_code in state_prod:
            col = f"{st_code}_GenerationIntensity"
            df_de = df_de.merge(state_prod[st_code][["timestamp", "Production_Intensity"]].rename(columns={"Production_Intensity": col}), on="timestamp", how="left")
        if st_code in state_cons:
            col = f"{st_code}_ConsumptionIntensity"
            df_de = df_de.merge(state_cons[st_code][["timestamp", "Consumption_Intensity"]].rename(columns={"Consumption_Intensity": col}), on="timestamp", how="left")

    # -1 hour shift for all intensity columns (like in pipeline)
    for col in df_de.columns:
        if "Intensity" in col and col != "timestamp":
            df_de[col] = df_de[col].shift(-1)

    # Final fill
    df_de.sort_values("timestamp", inplace=True)
    df_de.ffill(inplace=True)
    df_de.bfill(inplace=True)

    # Optionally, add Total Imports/Exports as in the end of the pipeline
    unique_import_cols = [col for col in df_de.columns if "imports" in col.lower() and col.startswith("DE_LU")]
    unique_export_cols = [col for col in df_de.columns if "exports" in col.lower() and col.startswith("DE_LU")]
    if unique_import_cols:
        df_de["Total_Imports"] = df_de[unique_import_cols].sum(axis=1, skipna=True)
    if unique_export_cols:
        df_de["Total_Exports"] = df_de[unique_export_cols].sum(axis=1, skipna=True)
    cols_to_drop = [col for col in df_de.columns
                    if (("imports" in col.lower() or "exports" in col.lower())
                        and not col.startswith("DE_LU"))]
    df_de.drop(columns=cols_to_drop, inplace=True)

    # ========== Guarantee required columns with default values if missing ==========
    col1 = "CH_query_query_generation_forecast_Actual_Aggregated"
    col2 = "t_10YNO_2_T_query_query_wind_and_solar_forecast_Wind_Offshore"
    if col1 not in df_de.columns:
        df_de[col1] = 3677
    if col2 not in df_de.columns:
        df_de[col2] = 0

    # Write to CSV
    consolidated_csv = os.path.join(csv_folder, "DE_LU_consolidated.csv")
    df_de.to_csv(consolidated_csv, index=False)
    print(
        f"[INFO] Wrote final cleaned data to {consolidated_csv} with shape={df_de.shape} "
        f"columns={df_de.columns.tolist()}"
    )

if __name__ == "__main__":
    main()

Fetching imports: CH to DE_LU...
Fetching exports: DE_LU to CH...
Fetching imports: DE_LU to CH...
Fetching exports: CH to DE_LU...
Fetching imports: DE_LU to BE...
Fetching exports: BE to DE_LU...
Fetching imports: BE to DE_LU...
Fetching exports: DE_LU to BE...
Fetching imports: DE_LU to CZ...
Fetching exports: CZ to DE_LU...
Fetching imports: CZ to DE_LU...
Fetching exports: DE_LU to CZ...
Fetching imports: DE_LU to FR...
Fetching exports: FR to DE_LU...
Fetching imports: FR to DE_LU...
Fetching exports: DE_LU to FR...
Fetching imports: NL to DE_LU...
Fetching exports: DE_LU to NL...
Fetching imports: DE_LU to NL...
Fetching exports: NL to DE_LU...
Fetching imports: PL to DE_LU...
Fetching exports: DE_LU to PL...
Fetching imports: DE_LU to PL...
Fetching exports: PL to DE_LU...
Fetching imports: DE_LU to DK_1...
Fetching exports: DK_1 to DE_LU...
Fetching imports: DK_1 to DE_LU...
Fetching exports: DE_LU to DK_1...
Fetching imports: DE_LU to DK_2...
Fetching exports: DK_2 to DE_LU..

In [ ]:
from autogluon.tabular import TabularPredictor
import os
import pandas as pd

def create_features(df: pd.DataFrame, label_col: str) -> pd.DataFrame:
    drop_cols = []
    for c in df.columns:
        if c in ["timestamp", "item_id"] or c.startswith("y_") or ("Intensity" in c and c != label_col):
            drop_cols.append(c)
    return df.drop(columns=drop_cols, errors="ignore")

def run_inference_on_consolidated_csv():
    # ====== Setup Paths ======
    csv_folder = "/content/drive/My Drive/Colab Notebooks/entsoe_pipeline_optimized123/entsoe_DB_data_DE"
    models_folder = "/content/drive/My Drive/Colab Notebooks/entsoe_pipeline_optimized123/autogluon_models"
    consolidated_csv = os.path.join(csv_folder, "DE_LU_consolidated.csv")

    # ====== Load Data ======
    df = pd.read_csv(consolidated_csv, parse_dates=["timestamp"])
    df.sort_values("timestamp", inplace=True)
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    # ====== DE-level Predictions ======
    y_cols = []  # collect new y-pred column names for reporting
    for intensity in ["Generation_Intensity", "Consumption_Intensity"]:
        model_dir = os.path.join(models_folder, f"DE_{intensity.replace('Intensity', 'Model')}")
        if not os.path.exists(model_dir):
            print(f"[WARN] Model not found: {model_dir}. Skipping.")
            continue

        predictor = TabularPredictor.load(model_dir, require_version_match=False)
        features = create_features(df, intensity)
        y_pred = predictor.predict(features)
        y_col = f"y_DE_{intensity}"
        df[y_col] = y_pred
        y_cols.append(y_col)
        print(f"[INFO] Prediction complete: {y_col}")

    # ====== State-level Predictions ======
    for col in df.columns:
        if col.endswith("_GenerationIntensity") or col.endswith("_ConsumptionIntensity"):
            if col.startswith("DE") or col == "Generation_Intensity" or col == "Consumption_Intensity":
                continue  # already handled
            st_code = col.split("_")[0]
            typ = "Generation" if "Generation" in col else "Consumption"
            model_dir = os.path.join(models_folder, f"{st_code}_{typ}_Model")
            if not os.path.exists(model_dir):
                print(f"[WARN] State Model not found: {model_dir}. Skipping.")
                continue
            predictor = TabularPredictor.load(model_dir, require_version_match=False)
            features = create_features(df, col)
            y_pred = predictor.predict(features)
            y_col = f"y_{st_code}_{typ}Intensity"
            df[y_col] = y_pred
            y_cols.append(y_col)
            print(f"[INFO] Prediction complete: {y_col}")

    # ====== Save New CSV ======
    pred_csv = os.path.join(csv_folder, "Predicted_Values_consolidated_DE_LU.csv")
    df.to_csv(pred_csv, index=False)
    print(f"[INFO] Saved prediction file: {pred_csv}")
    print(f"[INFO] Prediction columns added: {y_cols}")
    return df
df_1 = None
if __name__ == "__main__":
    df_1 = run_inference_on_consolidated_csv()
    # run_inference_on_consolidated_csv()

[INFO] Prediction complete: y_DE_Generation_Intensity



############################## WARNING ##############################
	Predictor Version: 1.2
	Current Version:   1.3.1
############################## WARNING ##############################

Found 1 mismatches between original and current metadata:


[INFO] Prediction complete: y_DE_Consumption_Intensity



############################## WARNING ##############################
	Predictor Version: 1.2
	Current Version:   1.3.1
############################## WARNING ##############################

Found 1 mismatches between original and current metadata:


[INFO] Prediction complete: y_BB_GenerationIntensity



############################## WARNING ##############################
	Predictor Version: 1.2
	Current Version:   1.3.1
############################## WARNING ##############################

Found 1 mismatches between original and current metadata:


[INFO] Prediction complete: y_BB_ConsumptionIntensity



############################## WARNING ##############################
	Predictor Version: 1.2
	Current Version:   1.3.1
############################## WARNING ##############################

Found 1 mismatches between original and current metadata:


[INFO] Prediction complete: y_BW_GenerationIntensity
[INFO] Prediction complete: y_BW_ConsumptionIntensity
[INFO] Saved prediction file: /content/drive/My Drive/Colab Notebooks/entsoe_pipeline_optimized123/entsoe_DB_data_DE/Predicted_Values_consolidated_DE_LU.csv
[INFO] Prediction columns added: ['y_DE_Generation_Intensity', 'y_DE_Consumption_Intensity', 'y_BB_GenerationIntensity', 'y_BB_ConsumptionIntensity', 'y_BW_GenerationIntensity', 'y_BW_ConsumptionIntensity']


In [ ]:
from sklearn.metrics import r2_score
import numpy as np

def calculate_metrics(df):
    results = []
    for col in df.columns:
        if col.startswith("y_"):
            # DE-level predictions: y_DE_Generation_Intensity
            if col.startswith("y_DE_"):
                actual_col = col.replace("y_DE_", "")  # Remove y_DE_ for DE
            else:
                actual_col = col.replace("y_", "")     # For state columns
            if actual_col in df.columns:
                actual = df[actual_col].astype(float)
                pred = df[col].astype(float)
                mask = ~np.isnan(actual) & ~np.isnan(pred)
                actual = actual[mask]
                pred = pred[mask]
                rmse = np.sqrt(np.mean((actual - pred) ** 2))
                mape = np.mean(np.abs((actual - pred) / (actual + 1e-8))) * 100
                smape = 100 * np.mean(2 * np.abs(pred - actual) / (np.abs(actual) + np.abs(pred) + 1e-8))
                r2 = r2_score(actual, pred)
                results.append({
                    "Target": actual_col,
                    "Prediction": col,
                    "RMSE": rmse,
                    "MAPE (%)": mape,
                    "SMAPE (%)": smape,
                    "R2": r2
                })
    return pd.DataFrame(results)

# ====== Add this before saving the new CSV ======
metrics_df = calculate_metrics(df_1)
metrics_csv = os.path.join(csv_folder, "Prediction_Metrics_DE_LU.csv")
metrics_df.to_csv(metrics_csv, index=False)
print(f"[INFO] Saved prediction metrics: {metrics_csv}")
print(metrics_df)

[INFO] Saved prediction metrics: /content/drive/My Drive/Colab Notebooks/entsoe_pipeline_optimized123/entsoe_DB_data_DE/Prediction_Metrics_DE_LU.csv
                    Target                  Prediction        RMSE   MAPE (%)  \
0     Generation_Intensity   y_DE_Generation_Intensity   46.272006  11.931514   
1    Consumption_Intensity  y_DE_Consumption_Intensity   41.822933  11.190004   
2   BB_GenerationIntensity    y_BB_GenerationIntensity   89.134603  15.531810   
3  BB_ConsumptionIntensity   y_BB_ConsumptionIntensity   87.992900  15.527673   
4   BW_GenerationIntensity    y_BW_GenerationIntensity  104.873899  29.946841   
5  BW_ConsumptionIntensity   y_BW_ConsumptionIntensity   63.278977  27.149866   

   SMAPE (%)        R2  
0  11.572635  0.896904  
1  11.232074  0.840353  
2  14.751431  0.731558  
3  14.701108  0.738332  
4  30.303370  0.635730  
5  26.350047  0.617189  


In [ ]:
df_1

,timestamp,DE_LU_imports_CH_0,DE_LU_exports_CH_0,DE_LU_imports_BE_0,DE_LU_exports_BE_0,DE_LU_imports_CZ_0,DE_LU_exports_CZ_0,DE_LU_imports_FR_0,DE_LU_exports_FR_0,DE_LU_imports_NL_0,...,BW_GenerationIntensity,BW_ConsumptionIntensity,CH_query_query_generation_forecast_Actual_Aggregated,t_10YNO_2_T_query_query_wind_and_solar_forecast_Wind_Offshore,y_DE_Generation_Intensity,y_DE_Consumption_Intensity,y_BB_GenerationIntensity,y_BB_ConsumptionIntensity,y_BW_GenerationIntensity,y_BW_ConsumptionIntensity
0,2025-04-01 00:00:00,0.000,1200.000,765.9,0.0,519.6,0.0,905.8,0.0,0.0,...,561.0,427.0,3677,0,560.699829,450.763702,664.124634,676.243225,440.423035,349.639679
1,2025-04-01 01:00:00,0.000,1125.000,824.3,0.0,603.4,0.0,1099.5,0.0,0.0,...,561.0,427.0,3677,0,571.199036,439.217773,696.101990,722.848022,450.881042,328.469543
2,2025-04-01 02:00:00,0.000,1644.200,659.4,0.0,993.7,0.0,1170.7,0.0,0.0,...,558.0,422.0,3677,0,580.854370,442.710052,700.272766,734.128296,431.371307,315.359161
3,2025-04-01 03:00:00,0.000,1416.100,407.3,0.0,991.7,0.0,1286.2,0.0,0.0,...,541.0,387.0,3677,0,574.736267,438.565399,700.583679,731.181274,434.510040,317.502808
4,2025-04-01 04:00:00,0.000,1369.500,929.9,0.0,941.9,0.0,1146.3,0.0,0.0,...,528.0,343.0,3677,0,585.737549,466.097595,698.748474,736.878113,449.463867,328.644592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
622,2025-04-26 22:00:00,92.744,25.000,0.0,319.9,0.0,508.3,1171.3,0.0,0.0,...,158.0,125.0,3677,0,301.555725,265.276947,390.016937,439.704987,225.292984,178.537552
623,2025-04-26 23:00:00,0.000,320.457,0.0,343.4,0.0,324.0,818.1,0.0,0.0,...,117.0,91.0,3677,0,283.550262,256.938538,447.435394,477.321259,213.851105,189.988510
624,2025-04-27 00:00:00,0.000,320.457,0.0,343.4,0.0,324.0,818.1,0.0,0.0,...,91.0,71.0,3677,0,276.455688,239.182419,442.955017,454.355560,230.340408,181.617401
625,2025-04-27 01:00:00,0.000,320.457,0.0,343.4,0.0,324.0,818.1,0.0,0.0,...,90.0,67.0,3677,0,276.455688,239.182419,442.955017,454.355560,230.340408,181.617401
